# Sunflower 14B GRPO Combined — Inference Notebook

This notebook runs inference on the `jq/sunflower-14b-bs64-lr1e-4-250919` base model combined with the `jq/sunflower-14b-grpo-combined` LoRA adapter, using **Unsloth + vLLM** for fast generation. It demonstrates:

1. Environment and dependency setup
2. Loading the base model and LoRA adapter
3. Running translation prompts across Ugandan languages (Luganda, Runyankole, Lusoga, Acholi, Ateso)
4. Running general-knowledge and reasoning prompts

## Setup / Dependency Installation

The commented commands below reinstall PyTorch, Transformers, vLLM, Unsloth and related CUDA 13.0 wheels. Uncomment and run once per fresh environment.

In [ ]:
# !uv pip install --force-reinstall --no-cache-dir \
#   torchaudio torchvision torchcodec \
#   --index-url https://download.pytorch.org/whl/cu130

# !uv pip install --force-reinstall transformers torch accelerate vllm unsloth huggingface_hub bitsandbytes "xformers==0.0.33.post1" --torch-backend=cu130

# uv pip install --force-reinstall https://github.com/vllm-project/vllm/releases/download/v0.17.1/vllm-0.17.1+cu130-cp38-abi3-manylinux_2_35_x86_64.whl

## Imports and Environment Flags

- `FastLanguageModel` (Unsloth) loads the base model with vLLM fast-inference enabled.
- `LLM` / `SamplingParams` / `LoRARequest` come from vLLM for generation and LoRA hot-swapping.
- `UNSLOTH_VLLM_STANDBY=0` avoids a known crash when standby mode is enabled.
- `FLASHINFER_DISABLE_VERSION_CHECK=1` skips a strict FlashInfer version check.

In [1]:
import os
from unsloth import FastLanguageModel
import torch
from huggingface_hub import snapshot_download, login
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

os.environ["UNSLOTH_VLLM_STANDBY"] = "0" # Causes crashes when set to 1
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] ="1"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Authenticate with Hugging Face

Required because the base model and adapter repositories are gated. Prompts for an access token interactively.

In [2]:
login()

## Model Paths and System Prompt

- `BASE_MODEL_PATH` — the SFT base checkpoint.
- `GRPO_LORA_PATH` — the GRPO-trained LoRA adapter merged at inference time.
- `SYSTEM_MESSAGE` — the Sunflower persona prompt prepended to every chat request.

In [3]:
BASE_MODEL_PATH = "jq/sunflower-14b-bs64-lr1e-4-250919"
GRPO_LORA_PATH = "jq/sunflower-14b-grpo-combined"
SYSTEM_MESSAGE = """You are Sunflower, a helpful assistant made by Sunbird AI who understands all Ugandan languages. You specialise in accurate translations, 
explanations, summaries and other language tasks."""

## Download the GRPO LoRA Adapter

`snapshot_download` pulls the full adapter repository (weights + `adapter_config.json`) to a local cache directory. The resulting path is passed to `LoRARequest` later.

In [4]:
model_lora_path = snapshot_download(repo_id=GRPO_LORA_PATH)

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/514M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/7.70k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

## Inference with Base Model

In [5]:
max_seq_length = 1024
lora_rank = 32 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_PATH,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)


WARNING 07-08 12:53:06 [vllm.py:1403] Current vLLM config is not set.
INFO 07-08 12:53:06 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 07-08 12:53:07 [vllm_utils.py:739] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.7.1: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.13.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading jq/sunflower-14b-bs64-lr1e-4-250919 with actual GPU utilization = 88.91%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.25 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 112.
Unsloth: v

/venv/main/lib/python3.12/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 07-08 12:53:40 [model.py:514] Resolved architecture: Qwen3ForCausalLM
INFO 07-08 12:53:40 [model.py:1661] Using max model len 1024
INFO 07-08 12:53:40 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-08 12:53:41 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='jq/sunflower-14b-bs64-lr1e-4-250919', speculative_config=None, tokenizer='jq/sunflower-14b-bs64-lr1e-4-250919', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reason

/venv/main/lib/python3.12/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 07-08 12:53:42 [topk_topp_sampler.py:47] Using FlashInfer for top-p & top-k sampling.
INFO 07-08 12:53:42 [gpu_model_runner.py:3562] Starting to load model jq/sunflower-14b-bs64-lr1e-4-250919...
INFO 07-08 12:53:42 [cuda.py:315] Using AttentionBackendEnum.FLASHINFER backend.


...

INFO 07-08 12:58:33 [core.py:259] init engine (profile, create kv cache, warmup model) took 244.61 seconds
INFO 07-08 12:58:35 [llm.py:360] Supported tasks: ('generate',)


`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: Just some info: will skip parsing ['norm', 'post_attention_layernorm', 'pre_feedforward_layernorm', 'q_norm', 'norm1', 'ffn_norm', 'post_feedforward_layernorm', 'post_per_layer_input_norm', 'k_norm', 'layer_norm1', 'input_layernorm', 'post_layernorm', 'attention_norm', 'layer_norm2', 'norm2']


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['norm', 'post_attention_layernorm', 'pre_feedforward_layernorm', 'q_norm', 'norm1', 'cross_attn_post_attention_layernorm', 'ffn_norm', 'post_feedforward_layernorm', 'post_per_layer_input_norm', 'k_norm', 'layer_norm1', 'input_layernorm', 'post_layernorm', 'cross_attn_input_layernorm', 'attention_norm', 'layer_norm2', 'norm2']


### Example instruction (English → Luganda translation)

In [6]:
instruction  = "Translate to luganda: I am watching an Arsenal game right now"

### Build the chat prompt and sampling configuration

The tokenizer's chat template formats the system + user messages into the model's expected prompt format. `SamplingParams` controls decoding: temperature 0.6, top-k 50, up to 1024 generated tokens.

In [7]:
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user",   "content": instruction},
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
sampling_params = SamplingParams(
    temperature = 0.6,
    top_k = 50,
    max_tokens = 1024,
)


### Run generation with the LoRA adapter attached

`fast_generate` routes the request through vLLM. `LoRARequest` tells vLLM to dynamically load and apply the downloaded adapter for this call — the base weights stay untouched, so adapters can be swapped per request.

In [8]:
output = model.fast_generate(
    [prompt],
    sampling_params = sampling_params,
    lora_request = LoRARequest(
            lora_name="adapter",   # any unique name/identifier
            lora_int_id=1,            # unique integer ID
            lora_path=model_lora_path,  # local dir with adapter_config.json etc.
        ),
)



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

WARNING 07-08 13:01:12 [input_processor.py:250] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

### Inspect raw output and extract the generated text

`output` is a list of vLLM `RequestOutput` objects; `output[0].outputs[0].text` is the decoded completion string.

In [9]:
output

[RequestOutput(request_id=0, prompt='<|im_start|>system\nYou are Sunflower, a helpful assistant made by Sunbird AI who understands all Ugandan languages. You specialise in accurate translations, \nexplanations, summaries and other language tasks.<|im_end|>\n<|im_start|>user\nTranslate to luganda: I am watching an Arsenal game right now<|im_end|>\n<|im_start|>assistant\n', prompt_token_ids=[151644, 8948, 198, 2610, 525, 8059, 38753, 11, 264, 10950, 17847, 1865, 553, 8059, 22592, 15235, 879, 30769, 678, 46330, 82808, 15459, 13, 1446, 3281, 1064, 304, 13382, 36693, 11, 715, 327, 10393, 804, 11, 68922, 323, 1008, 4128, 9079, 13, 151645, 198, 151644, 872, 198, 27473, 311, 53410, 9817, 25, 358, 1079, 10099, 458, 32002, 1809, 1290, 1431, 151645, 198, 151644, 77091, 198], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='Nindaba omupiira gwa Arsenal kati.', token_ids=[45, 484, 12004, 7861, 454, 72, 8832, 342, 9991, 32002, 595, 93

In [11]:
output[0]

RequestOutput(request_id=0, prompt='<|im_start|>system\nYou are Sunflower, a helpful assistant made by Sunbird AI who understands all Ugandan languages. You specialise in accurate translations, \nexplanations, summaries and other language tasks.<|im_end|>\n<|im_start|>user\nTranslate to luganda: I am watching an Arsenal game right now<|im_end|>\n<|im_start|>assistant\n', prompt_token_ids=[151644, 8948, 198, 2610, 525, 8059, 38753, 11, 264, 10950, 17847, 1865, 553, 8059, 22592, 15235, 879, 30769, 678, 46330, 82808, 15459, 13, 1446, 3281, 1064, 304, 13382, 36693, 11, 715, 327, 10393, 804, 11, 68922, 323, 1008, 4128, 9079, 13, 151645, 198, 151644, 872, 198, 27473, 311, 53410, 9817, 25, 358, 1079, 10099, 458, 32002, 1809, 1290, 1431, 151645, 198, 151644, 77091, 198], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='Nindaba omupiira gwa Arsenal kati.', token_ids=[45, 484, 12004, 7861, 454, 72, 8832, 342, 9991, 32002, 595, 930

In [12]:
response = output[0].outputs[0].text

In [13]:
response

'Nindaba omupiira gwa Arsenal kati.'

## Inference with Lora Adapter using vllm

In [14]:
def generate_response(instruction, temperature=0.6):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": instruction},
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True, # Must add for generation
        tokenize = False,
    )
    sampling_params = SamplingParams(
        temperature = temperature,
        top_k = 50,
        max_tokens = 1024,
    )
    output = model.fast_generate(
        [prompt],
        sampling_params = sampling_params,
        lora_request = LoRARequest(
                lora_name="adapter",   # any unique name/identifier
                lora_int_id=1,            # unique integer ID
                lora_path=model_lora_path,  # local dir with adapter_config.json etc.
            ),
    )
    response = output[0].outputs[0].text
    return response

### Helper: `generate_response(instruction)`

Wraps prompt formatting, sampling, and LoRA-enabled vLLM generation into a single reusable function used by the test cells below.

### Translation tests across Ugandan languages
The following cells translate the same English sentence into Luganda, Runyankole, Lusoga, Acholi, and Ateso to spot-check cross-lingual quality.

In [16]:
instruction = "Translate to luganda: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.7)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Omuwanguzi w'akamalirizo k'ekikopo kya Carabao Cup eya Ssande ajja kusobola okwesogga oluzannya lwa Europa Conference League sizoni eddako."

In [17]:
instruction = "Translate to runyakole: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.1)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Orikusinga empaka za Carabao Cup ez'akamaririzo ez'orw'eizooba rya Sande, naija kuba aine obugabe bw'okuzaana omu mpaka za Europa Conference League omu siizoni erikwija."

In [18]:
instruction = "Translate to lusoga: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.3)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Omuwanguzi w'omupiira gwa Carabao Cup ogw'olwomukaaga lwa Sande alifuna obukumu okwetaba mu mpaka dha Europa Conference League play-off round sizoni eidha."

In [19]:
instruction = "Translate to acholi: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.3)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'Laloc me tuku me agiki me Carabao Cup I ceng cabit binongo gum me cito I pyem tuku me Europa Conference League I kare me tuku mabino.'

In [20]:
instruction = "Translate to ateso: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.1)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'Etunganan loetelekaari abolia nakawasia naka Eikopo loka Carabao Naesabiti edumuni alomar toma abolia naka Europa Conference League kapak naka abolia naetupakini.'

## General Knowledge Q&A

Non-translation prompts to probe the model's open-domain knowledge and its grounding in Sunbird AI / Sunflower identity.

In [21]:
instruction = "Who is Sunbird AI, what do they do?"
res = generate_response(instruction)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'Sunbird AI is a non-profit AI research lab that focuses on building practical, open, and accessible AI systems.'

In [22]:
instruction = "What is Sunflower"
res = generate_response(instruction)
res

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'I am Sunflower, a language assistant created by **Sunbird AI**. I have been trained to understand and respond to text in **Acholi, Rutooro, Runyankore, and Lusoga**. I can also speak **English and Swahili**. I was created to be a **helpful language assistant**! 🌻👋 🤍👋👋 🤍👋👋👋 🤍👋👋👋👋 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp 🤍HandsUp

In [23]:
from IPython.display import display, Markdown

### Pretty-print responses as Markdown

`display(Markdown(res))` renders formatting (lists, bold, etc.) inline for easier review of longer answers.

In [ ]:
display(Markdown(res))

In [24]:
instruction = "Uganda yafuna ddi ebwetogoze?"
res = generate_response(instruction)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Uganda yafuna obwetwaze okuva ku Bungereza nga 9 Okitobba 1962, oluvannyuma lw'okulwanirira okw'amaanyi okw'eddembe ly'eggwanga n'okwefuga.

### Ugandan-language questions and long-form translations

Mixes Luganda-language questions (understanding + response in Luganda), knowledge cut-off probes, and longer English→Luganda passages to stress-test fluency on extended context.

In [25]:
instruction = "Uganda yakabera neba pulezidenti bameka, era bebani?"
res = generate_response(instruction)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Uganda ebadde n'abakulembeze b'eggwanga 5, okuva mu 1962, nga bano be bano:

- **Idi Amin** (1971–1979)
- **Tito Okello** (1979–1980)
- **Yoweri Museveni** (1986–okutuusa leero)
- **Godfrey Binaisa** (1971–1979)
- **Tito Okello** (1980–1985)

In [26]:
instruction = "When is Luwum day is Uganda?"
res = generate_response(instruction)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Luwum day in Uganda is observed on **March 16th** every year, in honor of the **late Archbishop Janani Luwum**.

In [27]:
instruction = """Translate to Luganda: The English Football League Cup, often referred to as the League Cup and officially known as the Carabao Cup for sponsorship reasons,
is an annual knockout competition in men's domestic football in England"""
res = generate_response(instruction, temperature=0.1)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

English Football League Cup, emirundi egisinga eyogerwako nga League Cup era emanyiddwa mu butongole nga Carabao Cup olw'ensonga z'okuvujjirirwa,
empaka ez'okuvuganya ezibaawo buli mwaka mu mupiira gw'abasajja ogw'omunda mu Bungereza.

In [28]:
instruction = """What is the current weather now?"""
res = generate_response(instruction)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

I am unable to access real-time data, but I can help you understand how to check the current weather in Uganda! 🌻

You can use a weather app or website, or ask a local person for the latest updates.

In [29]:
instruction = """What was the result from the Caraboa cup final of Arsenal Vs Manchester City?"""
res = generate_response(instruction)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In the 2015 Carabao Cup final, Arsenal lost 5-3 to Manchester City on penalties after a 1-1 draw in the 120 minutes of the game.

In [30]:
instruction = """When is Arsenal playing Sporting Lisbon in the Champions league"""
res = generate_response(instruction)
display(Markdown(res))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

I'm sorry, I don't have the current sports schedule, but I can help you with the date of the next match if you let me know!